# MongoDB Handling

After installing the MongoDB server in your machine, you can use this notebook for handling the initial processes with the database.

Specifically, in this step, we utilize Python's `pymongo` library to exploit its capabilities for MongoDB server interaction.

**Important Note: Be sure that the MongoDB server is up and running as a service in the background.**

For example, in macOS, to run MongoDB (i.e. the mongod process) as a service, run:

* `brew services start mongodb-community`

To stop a mongod running as a macOS service, use the following command as needed:

* `brew services stop mongodb-community`

To install MongoDB in your system, follow the instructions here:

* https://www.mongodb.com/docs/manual/administration/install-community/


**Note:** You can modify any of the processes below, however, you have to explain your thoughts.

In [ ]:
# import library for various processes with the OS
import os

## Load configuration

In [ ]:
# import library for yaml handling
import yaml

In [ ]:
config_path = os.path.join(os.getcwd(), "config.yml")

with open(config_path) as file:
    config = yaml.load(file, Loader=yaml.FullLoader)

## MongoDB database instantiation

The relevant information for the MongoDB client connection, the database name, and collection name is located in the configuration file.

```
# DB Connection with the uri (host)
client: "mongodb://localhost:27017/"

# db name
db: "aiot_course"

# db collection
col: "NAME YOUR COLLECTION"
```

In [ ]:
# import library for hanlding the MongoDB client
import pymongo
# import library for retrieving datetime
from datetime import datetime

### Create the database

To create a database in MongoDB, start by creating a MongoClient object, then specify a connection URL with the correct ip address and the name of the database you want to create.

MongoDB will create the database if it does not exist, and make a connection to it.

In [ ]:
client = pymongo.MongoClient(config["client"])

In [ ]:
db = client[config["db"]]

### Instantiate the collection

To create a collection in MongoDB, use database object and specify the name of the collection you want to create.

MongoDB will create the collection if it does not exist.

In [ ]:
col = db[config["col"]]

Initially, no collection will be shown in MongoDB before you enter the first document!

## Create the data collection

Uploading the gathered data to MongoDB collection. The data directory structure should be as follows:

```
.
└── data/
    ├── class_A/
    │   ├── data_A_01.csv
    │   ├── data_A_02.csv
    │   └── ..
    ├── class_B/
    │   ├── data_B_01.csv
    │   ├── data_B_02.csv
    │   └── .
    └── class ...
```

In [ ]:
# import library for hanlding the csv data and transformations
import pandas as pd
import json

Get data path:

In [ ]:
data_path = os.path.join(os.getcwd(), "data")
print(data_path)

List all files in a path:

In [ ]:
classes_folders_list = [f for f in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, f))]
print(classes_folders_list)

In [ ]:
# print files in folder
folder_path = os.path.join(data_path, classes_folders_list[0])
files_in_folder = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
print(files_in_folder)

Each document in the MongoDB database should have the following schema:

```json
{
  "data": {
    "acc_x": ["array", "of", "values"],
    "acc_y": ["array", "of", "values"],
    "acc_z": ["array", "of", "values"],
  },
  "label": "The label of the instance",
  "datetime": "MongoDB datetime object (it can be generated with the datetime.datetime.now() function"
}
```

Accordingly, if you are using gyroscope or both accelerometer and gyroscope, the following order and naming of the sensor keys should be defined:

* for gyroscope: `gyr_x`, `gyr_y`, `gyr_z` for the three axes
* for accelerometer and gyroscope: `acc_x`, `acc_y`, `acc_z`, `gyr_x`, `gyr_y`, `gyr_z` for the six axes

**Note: Be careful, the document is mandatory to have the aforementioned schema, in order to argue and proceed with the rest of the processes later on, in data engineering, plotting, etc.**

In [ ]:
from utils import pair_sensor_files, merge_sensor_data

## Provide the code to upload the data to MongoDB

In [ ]:
data_path = os.path.join(os.getcwd(), "data")

# All subdirs
classes_folders_list = [f for f in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, f))]

for class_folder in classes_folders_list:
    folder_path = os.path.join(data_path, class_folder)
    # All files of folder
    files_in_folder = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]

    # Full paths for reading files
    full_paths = [os.path.join(folder_path, f) for f in files_in_folder]

    # Pair session files together (acc+gyro)
    sessionfile_pairs = pair_sensor_files(full_paths)

    for session_name, sessionFiles in sessionfile_pairs.items():
        print(f"Dataset: {session_name}")
        # print(f"  Gyroscope file: {sessionFiles['gyro']}")
        # print(f"  Accelerometer file: {sessionFiles['acc']}")
        gyro_df = pd.read_csv(os.path.join(folder_path, sessionFiles['gyro']))
        acc_df = pd.read_csv(os.path.join(folder_path, sessionFiles['acc']))
        merged_df = merge_sensor_data(acc_df, gyro_df)

        if merged_df.empty:
            print(f"[EMPTY MERGE] ACC: {sessionFiles['acc']} | GYRO: {sessionFiles['gyro']}")
            continue
        
        document = {
            "data": {
                "acc_x": merged_df["acc_x"].tolist(),
                "acc_y": merged_df["acc_y"].tolist(),
                "acc_z": merged_df["acc_z"].tolist(),
                "gyro_x": merged_df["gyro_x"].tolist(),
                "gyro_y": merged_df["gyro_y"].tolist(),
                "gyro_z": merged_df["gyro_z"].tolist(),
            },
            "label": f'{class_folder}',
            "datetime": datetime.now()
        }
        col.insert_one(document)



## NOTES: 
- Timestamps aren't saved in document
- All documents saved in same collection, no categorization based on class
- Label is class name
- Sampling: 100Hz, 16g/s 2000deg/s